# 08 — Baselines clássicos (TEA): SVM, Random Forest e XGBoost (protocolo revisado)

**O que mudou em relação à versão anterior**

| Antes | Agora | Motivo |
|---|---|---|
| GridSearch interno com `scoring='f1'`, sem `class_weight` no XGBoost | Busca de hiperparâmetros aninhada, métrica de seleção **AUC-ROC** | AUC é a métrica principal do TCC; o F1 é instável com folds pequenos |
| Folds recriados no notebook | **Mesmos folds externos do notebook 07**, só lidos e auditados | Todos os modelos da base precisam compartilhar os folds |
| Sem baseline trivial | `DummyClassifier` + teste com rótulos embaralhados | Mostrar o que é "acaso" nesta base |
| Resultados só em tabela | Tabela longa por base/modelo/fold/semente + previsões OOF salvas | Rastreabilidade para o notebook 12 |

**Protocolo**
- 5 folds externos agrupados por site (do notebook 07); cada fold inteiro vira teste uma vez.
- Busca interna: até 3 folds agrupados por site, **somente se cada divisão tiver as duas classes** (checado antes, célula 3). Se 3 não for viável, tenta 2; se nenhuma for, usa hiperparâmetros padrão **e registra isso** — nunca cria split aleatório.
- Grades: SVM (C × gamma), Random Forest (árvores × profundidade × max_features), XGBoost (estimadores × profundidade × learning rate), exatamente como no plano. `scale_pos_weight` e `StandardScaler` usam só o treino de cada divisão.
- Sete métricas por fold. Os folds são validação cruzada por site, **não** um conjunto de teste independente.

In [1]:
# 1. Projeto, módulo de avaliação e dataset (TEA / ABIDE I)
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

_raiz = Path.cwd().resolve()
for _c in (_raiz, *_raiz.parents):
    if (_c / 'src' / 'config.py').exists():
        _raiz = _c
        break
else:
    raise RuntimeError('Não encontrei src/config.py. Abra a pasta TCC_TEA_TDAH no VS Code.')
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src import avaliacao as av
criar_pastas()

NOME_BASE = 'TEA'
ROTULO_CASO = 'TEA'
ARQ_DATASET = PATHS['processed'] / 'dataset_abide_processado.pkl'
PASTA_RES = Path(PATHS['results'])
PASTA_MOD = Path(PATHS['models'])
PASTA_RES.mkdir(parents=True, exist_ok=True)
PASTA_MOD.mkdir(parents=True, exist_ok=True)

# 2. Parâmetros desta execução
GRADE_REDUZIDA = False       # True só se a grade completa estourar o tempo (documentar no TCC)
SEMENTES = [SEED]            # a busca interna e o treino são determinísticos dado a semente
N_PERMUTACOES = 3            # teste de rótulos embaralhados (por fold)
EXCLUIR_SITES = []           # sensibilidade: p.ex. ['Pittsburgh', 'WashU'] no TDAH (ver nota na célula 3)
MODELOS = ['SVM_RBF', 'Random_Forest', 'XGBoost']
if av.RAPIDO:
    N_PERMUTACOES = 1
    print('*** TCC_RAPIDO=1: grades e permutações mínimas (só para testar o notebook) ***')

d = av.carregar_dataset(ARQ_DATASET)
d = av.filtrar_sites(d, EXCLUIR_SITES)
SUFIXO = '' if not EXCLUIR_SITES else '_sem_' + '_'.join(s.lower() for s in EXCLUIR_SITES)

print(f'Base: {NOME_BASE} | X: {d["X"].shape} | ROIs: {av.n_rois_de(d["X"].shape[1])}')
print('Classes (0=controle, 1=%s):' % ROTULO_CASO, pd.Series(d['y']).value_counts().sort_index().to_dict())
print('Sites:', len(np.unique(d['sites'])), '| Folds externos:', sorted(np.unique(d['fold_id']).tolist()))
print('Intervalo de X (Fisher z esperado, não [-1,1]):', float(d['X'].min()), 'a', float(d['X'].max()))

Base: TEA | X: (871, 19900) | ROIs: 200
Classes (0=controle, 1=TEA): {0: 468, 1: 403}
Sites: 20 | Folds externos: [0, 1, 2, 3, 4]
Intervalo de X (Fisher z esperado, não [-1,1]): -1.0973010063171387 a 1.9218876361846924


In [2]:
# 3. Auditoria dos folds externos e viabilidade dos splits internos
print('FOLDS EXTERNOS')
display(av.resumo_folds(d))

print('\nSITE x CLASSE (sites com uma só classe aparecem com 0 em uma coluna)')
display(pd.crosstab(pd.Series(d['sites'], name='site'), pd.Series(d['y'], name='classe')))

print('\nVIABILIDADE DA BUSCA INTERNA AGRUPADA POR SITE (por fold externo)')
aud_int = av.auditoria_splits_internos(d, k_max=3)
display(aud_int)
if not aud_int['viavel'].all():
    print('ATENÇÃO: há fold externo sem partição interna viável; ali serão usados hiperparâmetros padrão (registrado).')
if (aud_int['k_usado'] < 3).any():
    print('Há fold(s) com menos de 3 folds internos; isso fica registrado na coluna k_interno dos resultados.')

FOLDS EXTERNOS


,fold,n_treino,n_teste,controles_teste,casos_teste,controles_treino,casos_treino,sites_teste,sobreposicao_sites,duas_classes_no_teste
0,0,830,41,19,22,449,381,YALE,0,True
1,1,698,173,101,72,367,331,"KKI, OLIN, SBL, UM_1",0,True
2,2,706,165,91,74,377,329,"CALTECH, LEUVEN_1, LEUVEN_2, PITT, TRINITY",0,True
3,3,547,324,175,149,293,254,"CMU, NYU, OHSU, SDSU, STANFORD, UCLA_1",0,True
4,4,703,168,82,86,386,317,"MAX_MUN, UCLA_2, UM_2, USM",0,True



SITE x CLASSE (sites com uma só classe aparecem com 0 em uma coluna)


classe,0,1
site,,
CALTECH,10,5
CMU,5,6
KKI,21,12
LEUVEN_1,14,14
LEUVEN_2,16,12
MAX_MUN,27,19
NYU,98,74
OHSU,13,12
OLIN,14,14



VIABILIDADE DA BUSCA INTERNA AGRUPADA POR SITE (por fold externo)


,fold_externo,k_pedido,n_sites_treino,k_usado,semente_particao,viavel,tam_validacoes
0,0,3,19,3,42,True,"[158, 496, 176]"
1,1,3,16,3,42,True,"[92, 264, 342]"
2,2,3,15,3,42,True,"[178, 177, 351]"
3,3,3,14,3,42,True,"[231, 170, 146]"
4,4,3,16,3,42,True,"[102, 170, 431]"


In [3]:
# 4. Treino e avaliação (nested: busca interna -> refit no treino externo -> teste no fold externo)
from sklearn.dummy import DummyClassifier
from sklearn.base import clone

folds = sorted(np.unique(d['fold_id']).tolist())
nomes_todos = ['Dummy_prior', 'Dummy_estratificado'] + MODELOS
proba_oof = {m: np.full(len(d['y']), np.nan, dtype=np.float32) for m in nomes_todos}
pred_oof = {m: np.full(len(d['y']), -1, dtype=np.int8) for m in nomes_todos}
linhas, tabelas_busca = [], []

for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]
    tr = np.where(d['fold_id'] != fold)[0]
    Xtr, ytr, Xte, yte = d['X'][tr], d['y'][tr], d['X'][te], d['y'][te]
    splits, info = av.splits_internos(ytr, d['sites'][tr], k_max=3, seed=SEED)
    print(f'\n=== Fold {fold}: treino={len(tr)} teste={len(te)} | folds internos: ' +
          (f"{info['k_usado']}" if info['viavel'] else 'inviável -> hiperparâmetros padrão'))

    # Dummies (referência de acaso)
    for nome, estrategia in [('Dummy_prior', 'prior'), ('Dummy_estratificado', 'stratified')]:
        dm = DummyClassifier(strategy=estrategia, random_state=SEED).fit(Xtr, ytr)
        p = dm.predict_proba(Xte)[:, 1]
        pr = dm.predict(Xte)
        proba_oof[nome][te], pred_oof[nome][te] = p, pr
        linhas.append(av.linha_resultado(NOME_BASE, nome, fold, SEED, yte, pr, p, k_interno=0, params='-'))

    for nome in MODELOS:
        for sem in SEMENTES:
            t0 = av.agora()
            if info['viavel']:
                melhores, tab = av.busca_hiperparametros(nome, Xtr, ytr, splits, seed=sem, reduzida=GRADE_REDUZIDA)
                tab.insert(0, 'fold_externo', fold); tab.insert(0, 'modelo', nome); tab.insert(0, 'semente', sem)
                tabelas_busca.append(tab)
            else:
                melhores = {}
            t_busca = av.agora() - t0
            modelo = av.criar_modelo(nome, ytr, seed=sem, params=melhores)
            modelo.fit(Xtr, ytr)               # refit com TODO o treino externo
            p = av.pontuar(modelo, Xte)
            pr = modelo.predict(Xte).astype(np.int8)
            if sem == SEMENTES[0]:
                proba_oof[nome][te], pred_oof[nome][te] = p, pr
            lin = av.linha_resultado(NOME_BASE, nome, fold, sem, yte, pr, p,
                                     k_interno=int(info['k_usado']), params=str(melhores),
                                     seg_busca=round(t_busca, 1), seg_total=round(av.agora() - t0, 1))
            linhas.append(lin)
            aviso = '  <-- ALERTA: previu uma única classe' if lin['classe_unica_prevista'] else ''
            print(f"  {nome:14s} sem={sem} AUC={lin['auc_roc']:.3f} BalAcc={lin['balanced_accuracy']:.3f} "
                  f"Rec={lin['recall']:.2f} Esp={lin['specificity']:.2f} | {t_busca/60:.1f} min de busca | {melhores}{aviso}")
    pd.DataFrame(linhas).to_csv(PASTA_RES / f'08_resultados_long{SUFIXO}.csv', index=False)   # checkpoint por fold

for m in nomes_todos:
    if np.any(pred_oof[m] < 0) or not np.isfinite(proba_oof[m]).all():
        raise RuntimeError(f'Previsões OOF incompletas para {m}.')
df_long = pd.DataFrame(linhas)
df_long['tipo'] = 'externo'
if tabelas_busca:
    pd.concat(tabelas_busca).to_csv(PASTA_RES / f'08_busca_hiperparametros{SUFIXO}.csv', index=False)


=== Fold 0: treino=830 teste=41 | folds internos: 3
  SVM_RBF        sem=42 AUC=0.792 BalAcc=0.785 Rec=0.73 Esp=0.84 | 9.0 min de busca | {'clf__C': 10, 'clf__gamma': 'scale'}
  Random_Forest  sem=42 AUC=0.739 BalAcc=0.678 Rec=0.41 Esp=0.95 | 3.7 min de busca | {'n_estimators': 500, 'max_depth': 10, 'max_features': 'sqrt'}
  XGBoost        sem=42 AUC=0.687 BalAcc=0.713 Rec=0.64 Esp=0.79 | 106.1 min de busca | {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1}

=== Fold 1: treino=698 teste=173 | folds internos: 3
  SVM_RBF        sem=42 AUC=0.687 BalAcc=0.614 Rec=0.62 Esp=0.60 | 5.0 min de busca | {'clf__C': 10, 'clf__gamma': 'scale'}
  Random_Forest  sem=42 AUC=0.666 BalAcc=0.633 Rec=0.65 Esp=0.61 | 3.0 min de busca | {'n_estimators': 500, 'max_depth': 20, 'max_features': 'sqrt'}
  XGBoost        sem=42 AUC=0.689 BalAcc=0.663 Rec=0.65 Esp=0.67 | 80.3 min de busca | {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1}

=== Fold 2: treino=706 teste=165 | folds internos: 

In [4]:
# 5. Sanidade: rótulos de treino embaralhados (a AUC deve ficar perto de 0,5)
# Usa o SVM com hiperparâmetros padrão (sem busca) para ser rápido. Se a AUC aqui ficar claramente
# acima de 0,5, há vazamento ou erro de pipeline — investigue antes de interpretar qualquer resultado.
rng = np.random.default_rng(SEED)
linhas_perm = []
for fold in folds:
    te = np.where(d['fold_id'] == fold)[0]
    tr = np.where(d['fold_id'] != fold)[0]
    for k in range(N_PERMUTACOES):
        y_emb = rng.permutation(d['y'][tr])
        m = av.criar_modelo('SVM_RBF', y_emb, seed=SEED).fit(d['X'][tr], y_emb)
        p = av.pontuar(m, d['X'][te])
        linhas_perm.append(dict(fold=fold, perm=k, auc=float(av.roc_auc_score(d['y'][te], p))))
df_perm = pd.DataFrame(linhas_perm)
print('AUC com rótulos embaralhados (SVM):  média = %.3f  | mín = %.3f  | máx = %.3f' %
      (df_perm.auc.mean(), df_perm.auc.min(), df_perm.auc.max()))
print('Esperado: ~0,5 (variação de ±0,05 a ±0,10 é normal em folds pequenos).')
df_perm.to_csv(PASTA_RES / f'08_teste_rotulos_embaralhados{SUFIXO}.csv', index=False)

AUC com rótulos embaralhados (SVM):  média = 0.499  | mín = 0.415  | máx = 0.603
Esperado: ~0,5 (variação de ±0,05 a ±0,10 é normal em folds pequenos).


In [5]:
# 6. Resumo: média por fold, média ponderada pelo tamanho, AUC OOF com IC por bootstrap de sites
res = av.resumir_resultados(df_long, d, proba_oof)
pd.set_option('display.width', 200)
print('MÉDIA ± DP ENTRE FOLDS (1 linha por modelo)')
display(res['por_fold'])
print('\nMÉDIA PONDERADA PELO TAMANHO DO FOLD')
display(res['ponderado'])
print('\nAUC OOF (previsões fora do treino reunidas) com IC 95% por bootstrap de sites')
display(res['auc_oof_ic'])
print('Critério definido a priori: IC inferior > 0,5 => "acima do acaso". Não é prova; com poucos sites o IC é largo.')
print('Não se usa Wilcoxon com 5 folds. Diferenças entre modelos são só reportadas, com sua incerteza.')

print('\nDESEMPENHO POR SITE (modelo com maior AUC OOF)')
melhor = res['auc_oof_ic'].sort_values('auc', ascending=False).iloc[0]['modelo']
tab_site = av.tabela_por_site(d, proba_oof[melhor], pred_oof[melhor])
print('Modelo:', melhor)
display(tab_site.round(3))

MÉDIA ± DP ENTRE FOLDS (1 linha por modelo)


accuracy         balanced_accuracy         precision          recall         specificity              f1         auc_roc        
                        mean     std              mean     std      mean     std    mean     std        mean     std    mean     std    mean     std
modelo                                                                                                                                              
Dummy_estratificado   0.4661  0.0299            0.4631  0.0307    0.4297  0.0596  0.3820  0.0577      0.5441  0.0202  0.4024  0.0483  0.4631  0.0307
Dummy_prior           0.5254  0.0488            0.5000  0.0000    0.0000  0.0000  0.0000  0.0000      1.0000  0.0000  0.0000  0.0000  0.5000  0.0000
Random_Forest         0.6200  0.0308            0.6140  0.0495    0.6887  0.1424  0.4104  0.1657      0.8176  0.1349  0.4900  0.1159  0.6833  0.0364
SVM_RBF               0.6654  0.0671            0.6629  0.0710    0.6609  0.1165  0.5981  0.0916      0.7277  0.0849  0.6251  0.0949  0.7241  0.0494
XGBoost               0.6574  0.0303            0.6531  0.0376    0.6636  0.0729  0.5580  0.0868      0.7482  0.0471  0.6030  0.0663  0.6907  0.0224


MÉDIA PONDERADA PELO TAMANHO DO FOLD


,modelo,accuracy_pond,balanced_accuracy_pond,precision_pond,recall_pond,specificity_pond,f1_pond,auc_roc_pond
0,Dummy_estratificado,0.4753,0.4708,0.4270,0.4028,0.5388,0.4132,0.4708
1,Dummy_prior,0.5373,0.5000,0.0000,0.0000,1.0000,0.0000,0.5000
2,Random_Forest,0.6062,0.5942,0.6353,0.3944,0.7940,0.4645,0.6689
3,SVM_RBF,0.6475,0.6436,0.6309,0.5773,0.7099,0.6003,0.7148
4,XGBoost,0.6464,0.6392,0.6429,0.5316,0.7468,0.5783,0.6887



AUC OOF (previsões fora do treino reunidas) com IC 95% por bootstrap de sites


,modelo,auc,ic_inf,ic_sup,n_boot_validos
0,Dummy_prior,0.4638,0.4203,0.5069,2000
1,Dummy_estratificado,0.4702,0.4343,0.5024,2000
2,SVM_RBF,0.7121,0.6640,0.7486,2000
3,Random_Forest,0.6532,0.6160,0.6891,2000
4,XGBoost,0.6781,0.6546,0.7013,2000


Critério definido a priori: IC inferior > 0,5 => "acima do acaso". Não é prova; com poucos sites o IC é largo.
Não se usa Wilcoxon com 5 folds. Diferenças entre modelos são só reportadas, com sua incerteza.

DESEMPENHO POR SITE (modelo com maior AUC OOF)
Modelo: SVM_RBF


,site,fold,n,controles,casos,acuracia,auc
0,CALTECH,2,15,10,5,0.533,0.500
1,CMU,3,11,5,6,0.727,0.900
2,KKI,1,33,21,12,0.606,0.758
3,LEUVEN_1,2,28,14,14,0.536,0.704
4,LEUVEN_2,2,28,16,12,0.643,0.797
5,MAX_MUN,4,46,27,19,0.478,0.546
6,NYU,3,172,98,74,0.715,0.792
7,OHSU,3,25,13,12,0.360,0.359
8,OLIN,1,28,14,14,0.643,0.684
9,PITT,2,50,26,24,0.740,0.753


In [6]:
# 7. Salvar tudo (tabela longa, resumos, OOF)
df_long.to_csv(PASTA_RES / f'08_resultados_long{SUFIXO}.csv', index=False)
res['por_fold'].to_csv(PASTA_RES / f'08_resumo_por_fold{SUFIXO}.csv')
res['ponderado'].to_csv(PASTA_RES / f'08_resumo_ponderado{SUFIXO}.csv', index=False)
res['auc_oof_ic'].to_csv(PASTA_RES / f'08_auc_oof_ic{SUFIXO}.csv', index=False)
tab_site.to_csv(PASTA_RES / f'08_por_site_{melhor}{SUFIXO}.csv', index=False)
av.salvar_oof(PASTA_RES / f'08_predicoes_oof{SUFIXO}.npz', d, proba_oof, pred_oof)
print('Salvo em:', PASTA_RES)
print('Obs.: não é salvo modelo "final" ajustado em todos os dados aqui; para XAI (notebook 11) o modelo é '
      'reajustado por fold com os hiperparâmetros registrados em 08_resultados_long.csv.')

Salvo em: /home/maria/TCC_TEA/Results
Obs.: não é salvo modelo "final" ajustado em todos os dados aqui; para XAI (notebook 11) o modelo é reajustado por fold com os hiperparâmetros registrados em 08_resultados_long.csv.
